# Employment Tribunal Judgment Calibration Pipeline

This notebook supports two judgment-input flows:

- **Debug**: process exactly one selected judgment PDF.
- **Batch**: process every PDF in a judgment folder.

The WS statement and controlled dictionary are shared inputs. They are loaded once per run. Only judgment PDFs vary inside the batch loop.

## 1. Execution Controls And Inputs

Start here. Keep `RUN_LLM = False` for dry runs. Set it to `True` only when ready to call the API.

For `RUN_MODE`, use:

- `"debug"` to select one file via `DEBUG_JUDGMENT_PATH`.
- `"batch"` to process all `*.pdf` files in `BATCH_JUDGMENTS_DIR`.

In [7]:
from pathlib import Path
import getpass
import importlib
import os
import sys

# === API EXECUTION SWITCH ===
RUN_LLM = True
SAVE_OUTPUTS = True
USE_CACHE = True
RUN_WS_TAGGING = True  # Runs once because WS and dictionary do not vary by judgment.

# === INPUT FLOW ===
RUN_MODE = "debug"  # "debug" or "batch"

case = "Ebeneezer_Paul_Tagoe_-vs-_Fernando_Private_Ltd_-_1403994.2023_-_Judgment.pdf"
PROJECT_ROOT = Path.cwd()
WS_PATH = PROJECT_ROOT / "input" / "ws" / "witness_statement.pdf"
DEBUG_JUDGMENT_PATH = PROJECT_ROOT / "input" / "judgments" / case
BATCH_JUDGMENTS_DIR = PROJECT_ROOT / "input" / "judgments"
DICTIONARY_PATH = PROJECT_ROOT / "input" / "dictionary" / "WS_Controlled_Theme_Dictionary_v1_2_final.json"

# === MODEL/API CONFIGURATION ===
API_PROVIDER = "openai"
MODEL_NAME = "gpt-5.5"  # Change only to a valid model available to your account.
MAX_COMPLETION_TOKENS = 12000

# === INPUT BOUNDS ===
MAX_INPUT_FILE_BYTES = 5_000_000
MAX_WS_CHARS = 80_000
WARN_JUDGMENT_CHARS = 120_000
MAX_JUDGMENT_CHARS = 250_000
MAX_DICTIONARY_THEMES = 20
MAX_REPAIR_ATTEMPTS = 3

# === DISPLAY BOUNDS ===
MAX_OUTPUT_PREVIEW_CHARS = 2_000
MAX_ERROR_PREVIEW = 10

SRC_PATH = str(PROJECT_ROOT / "src")
if SRC_PATH in sys.path:
    sys.path.remove(SRC_PATH)
sys.path.insert(0, SRC_PATH)

import config as config_module
import io_utils as io_utils_module
import text_extract as text_extract_module
import dictionary_loader as dictionary_loader_module
import dictionary_runner as dictionary_runner_module
import llm_client as llm_client_module
import calibration_runner as calibration_runner_module
import validators as validators_module
import repair_runner as repair_runner_module
import compression_runner as compression_runner_module

config_module = importlib.reload(config_module)
io_utils_module = importlib.reload(io_utils_module)
text_extract_module = importlib.reload(text_extract_module)
dictionary_loader_module = importlib.reload(dictionary_loader_module)
dictionary_runner_module = importlib.reload(dictionary_runner_module)
llm_client_module = importlib.reload(llm_client_module)
calibration_runner_module = importlib.reload(calibration_runner_module)
validators_module = importlib.reload(validators_module)
repair_runner_module = importlib.reload(repair_runner_module)
compression_runner_module = importlib.reload(compression_runner_module)

Config = config_module.Config
default_require_temperature_support = config_module.default_require_temperature_support
ensure_dirs = io_utils_module.ensure_dirs
read_text_file = io_utils_module.read_text_file
write_json = io_utils_module.write_json
write_text = io_utils_module.write_text
make_run_id = io_utils_module.make_run_id
make_source_slug = io_utils_module.make_source_slug
load_text = text_extract_module.load_text
load_dictionary = dictionary_loader_module.load_dictionary
validate_dictionary = dictionary_loader_module.validate_dictionary
run_ws_tagging = dictionary_runner_module.run_ws_tagging
build_ws_tagging_summary = dictionary_runner_module.build_ws_tagging_summary
LLMClient = llm_client_module.LLMClient
run_calibration = calibration_runner_module.run_calibration
CalibrationValidationContext = validators_module.CalibrationValidationContext
validate_calibration_output = validators_module.validate_calibration_output
repair_calibration_output = repair_runner_module.repair_calibration_output
run_compression = compression_runner_module.run_compression

REQUIRE_TEMPERATURE_SUPPORT = default_require_temperature_support(MODEL_NAME)  # GPT-5.x defaults to provider temperature

REQUIRED_ENV_BY_PROVIDER = {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "local": None,
}

required_env = REQUIRED_ENV_BY_PROVIDER.get(API_PROVIDER)
if RUN_LLM and required_env and not os.environ.get(required_env):
    os.environ[required_env] = getpass.getpass(f"{required_env}: ")

run_id = make_run_id()
config = Config.default(run_id)
config.run_mode = RUN_MODE
config.ws_path = WS_PATH
config.judgment_path = DEBUG_JUDGMENT_PATH
config.judgments_dir = BATCH_JUDGMENTS_DIR
config.dictionary_path = DICTIONARY_PATH
config.api_provider = API_PROVIDER
config.model_name = MODEL_NAME
config.max_tokens = MAX_COMPLETION_TOKENS
config.require_temperature_support = REQUIRE_TEMPERATURE_SUPPORT
config.cache_enabled = USE_CACHE
config.max_repair_attempts = MAX_REPAIR_ATTEMPTS

ensure_dirs(config)
judgment_paths = config.selected_judgment_paths()

print(f"Run ID: {run_id}")
print(f"RUN_LLM: {RUN_LLM}")
print(f"RUN_MODE: {config.run_mode}")
print(f"WS path: {config.ws_path}")
if config.run_mode == "debug":
    print(f"Debug judgment path: {config.judgment_path}")
else:
    print(f"Batch judgments dir: {config.judgments_dir}")
print(f"Judgments selected: {len(judgment_paths)}")
for selected_path in judgment_paths[:10]:
    print(f"  - {selected_path.name}")
print(f"Provider/model: {config.api_provider} / {config.model_name}")
print(f"Max completion tokens: {config.max_tokens}")
print(f"Require temperature support: {config.require_temperature_support}")
print(f"API key set: {bool(os.environ.get(required_env)) if required_env else 'not required'}")
print(f"Cache enabled: {config.cache_enabled}")


Run ID: 20260502_071343
RUN_LLM: True
RUN_MODE: debug
WS path: /home/hello/Projects/Calibrator/input/ws/witness_statement.pdf
Debug judgment path: /home/hello/Projects/Calibrator/input/judgments/Ebeneezer_Paul_Tagoe_-vs-_Fernando_Private_Ltd_-_1403994.2023_-_Judgment.pdf
Judgments selected: 1
  - Ebeneezer_Paul_Tagoe_-vs-_Fernando_Private_Ltd_-_1403994.2023_-_Judgment.pdf
Provider/model: openai / gpt-5.5
Max completion tokens: 12000
Require temperature support: False
API key set: True
Cache enabled: True


## 2. Input Checks

Common inputs are checked once. In batch mode, every selected judgment PDF is checked before any model call.

In [8]:
def require_file_within_limit(path: Path, max_bytes: int) -> None:
    if not path.exists():
        raise FileNotFoundError(path)
    size = path.stat().st_size
    if size > max_bytes:
        raise ValueError(f"{path} is {size:,} bytes, above limit {max_bytes:,}")

if RUN_MODE not in {"debug", "batch"}:
    raise ValueError('RUN_MODE must be "debug" or "batch"')
if not judgment_paths:
    raise ValueError(f"No judgment PDFs selected for RUN_MODE={RUN_MODE}")

common_input_paths = [
    config.ws_path,
    config.dictionary_path,
    config.ws_tagging_prompt_path,
    config.calibration_prompt_path,
    config.repair_prompt_path,
    config.compression_prompt_path,
]

for input_path in common_input_paths:
    require_file_within_limit(input_path, MAX_INPUT_FILE_BYTES)
for judgment_path in judgment_paths:
    require_file_within_limit(judgment_path, MAX_INPUT_FILE_BYTES)

print("Common inputs and selected judgment PDFs are within byte limits")

Common inputs and selected judgment PDFs are within byte limits


## 3. Shared Input Loading

The WS, controlled dictionary, prompts, and validation context are loaded once. In batch mode these are reused for every judgment.

In [9]:
ws_text = load_text(config.ws_path, config.cache_root / "text", config.cache_enabled)
if len(ws_text) > MAX_WS_CHARS:
    raise ValueError(f"WS text is {len(ws_text):,} chars, above limit {MAX_WS_CHARS:,}")

ws_tagging_prompt = read_text_file(config.ws_tagging_prompt_path)
calibration_prompt = read_text_file(config.calibration_prompt_path)
repair_prompt = read_text_file(config.repair_prompt_path)
compression_prompt = read_text_file(config.compression_prompt_path)

dictionary = load_dictionary(config.dictionary_path)
validate_dictionary(dictionary)
validation_context = CalibrationValidationContext.from_dictionary(dictionary)

theme_count = len(dictionary.get("ws_theme_dictionary", []))
if theme_count > MAX_DICTIONARY_THEMES:
    raise ValueError(f"Dictionary has {theme_count} themes, above limit {MAX_DICTIONARY_THEMES}")

print(f"WS text: {len(ws_text):,} chars")
print(f"Controlled dictionary themes: {theme_count}")
print("Prompts loaded")

WS text: 65,472 chars
Controlled dictionary themes: 20
Prompts loaded


## 4. LLM Client

In [10]:
llm_client = LLMClient(
    provider=config.api_provider,
    model=config.model_name,
    temperature=config.temperature,
    max_tokens=config.max_tokens,
    require_temperature_support=config.require_temperature_support,
    cache_dir=config.cache_root / "llm",
    cache_enabled=config.cache_enabled,
)

print(f"LLM client: {config.api_provider} / {config.model_name}")

LLM client: openai / gpt-5.5


## 5. Run Shared WS Tagging

This runs once per notebook run because the WS and dictionary are fixed across all judgments.

In [11]:
ws_tagging = None
ws_tagging_summary = None
if RUN_LLM and RUN_WS_TAGGING:
    ws_tagging = run_ws_tagging(ws_text, dictionary, ws_tagging_prompt, llm_client)
    ws_tagging_summary = build_ws_tagging_summary(ws_tagging)
    print(f"WS tagging completed with {len(ws_tagging.get('theme_mappings', []))} theme mappings")
    print(f"WS tagging summary themes: {len(ws_tagging_summary.get('theme_presence_by_id', {}))}")
elif RUN_LLM:
    print("WS tagging skipped")
else:
    print("RUN_LLM is False; skipped WS tagging")

WS tagging completed with 20 theme mappings
WS tagging summary themes: 20


## 6. Run Judgment Calibration

In debug mode this loop has one judgment. In batch mode it processes every selected PDF. The WS, dictionary, prompts, and validation context are reused.

In [12]:
results = []

if RUN_LLM:
    if SAVE_OUTPUTS:
        write_text(config.output_root / "extracted_text" / f"{run_id}_ws_text.txt", ws_text)
        if ws_tagging is not None:
            write_json(config.output_root / "ws_tagging" / f"{run_id}_ws_tagging.json", ws_tagging, validate_reload=config.validate_json_writes)
        if ws_tagging_summary is not None:
            write_json(config.output_root / "ws_tagging" / f"{run_id}_ws_tagging_summary.json", ws_tagging_summary, validate_reload=config.validate_json_writes)

    for judgment_path in judgment_paths:
        case_slug = make_source_slug(judgment_path)
        judgment_text = load_text(judgment_path, config.cache_root / "text", config.cache_enabled)
        judgment_chars = len(judgment_text)
        if judgment_chars > MAX_JUDGMENT_CHARS:
            raise ValueError(f"{judgment_path.name} text is {judgment_chars:,} chars, above hard limit {MAX_JUDGMENT_CHARS:,}")
        if judgment_chars > WARN_JUDGMENT_CHARS:
            print(f"Large judgment warning: {judgment_path.name} text is {judgment_chars:,} chars. This will increase token use.")

        calibration = run_calibration(
            ws_text,
            judgment_text,
            dictionary,
            ws_tagging_summary or {},
            calibration_prompt,
            llm_client,
        )
        errors = validate_calibration_output(
            calibration,
            context=validation_context,
            ws_tagging_summary=ws_tagging_summary or {},
        )
        validated_calibration = calibration
        repair_attempts = 0

        while errors and repair_attempts < config.max_repair_attempts:
            repair_attempts += 1
            if SAVE_OUTPUTS:
                write_json(config.output_root / "calibration_repaired" / f"{run_id}_{case_slug}_repair_attempt_{repair_attempts}.json", validated_calibration, validate_reload=config.validate_json_writes)
                write_json(config.output_root / "calibration_repaired" / f"{run_id}_{case_slug}_validation_errors_attempt_{repair_attempts}.json", errors, validate_reload=config.validate_json_writes)
            validated_calibration = repair_calibration_output(
                validated_calibration,
                errors,
                dictionary,
                ws_tagging_summary or {},
                repair_prompt,
                llm_client,
            )
            errors = validate_calibration_output(
                validated_calibration,
                context=validation_context,
                ws_tagging_summary=ws_tagging_summary or {},
            )

        if errors:
            print(f"Validation failed for {judgment_path.name} after {config.max_repair_attempts} repair attempts")
            for error in errors[:MAX_ERROR_PREVIEW]:
                print(error)
            raise ValueError(f"Calibration validation failed for {judgment_path.name}")

        reinforcement_plan = run_compression(validated_calibration, dictionary, compression_prompt, llm_client)

        output_files = {}
        if SAVE_OUTPUTS:
            extracted_judgment_path = config.output_root / "extracted_text" / f"{run_id}_{case_slug}_text.txt"
            raw_path = config.output_root / "calibration_raw" / f"{run_id}_{case_slug}_calibration_raw.json"
            validated_path = config.output_root / "calibration_validated" / f"{run_id}_{case_slug}_calibration_validated.json"
            reinforcement_path = config.output_root / "compression" / f"{run_id}_{case_slug}_reinforcement_plan.json"

            write_text(extracted_judgment_path, judgment_text)
            write_json(raw_path, calibration, validate_reload=config.validate_json_writes)
            write_json(validated_path, validated_calibration, validate_reload=config.validate_json_writes)
            write_json(reinforcement_path, reinforcement_plan, validate_reload=config.validate_json_writes)

            output_files = {
                "Extracted Judgment": extracted_judgment_path,
                "Raw Calibration": raw_path,
                "Validated Calibration": validated_path,
                "Reinforcement Plan": reinforcement_path,
            }

        results.append({
            "judgment_path": judgment_path,
            "case_slug": case_slug,
            "case_name": validated_calibration.get("case_metadata", {}).get("case_name", "Unknown"),
            "judgment_signals": len(validated_calibration.get("judgment_signals", [])),
            "repair_attempts": repair_attempts,
            "compressed_clusters": len(reinforcement_plan.get("compressed_reinforcement_plan", [])),
            "output_files": output_files,
        })
        print(f"Completed {judgment_path.name}")
else:
    print("RUN_LLM is False; skipped judgment calibration loop")

Large judgment warning: Ebeneezer_Paul_Tagoe_-vs-_Fernando_Private_Ltd_-_1403994.2023_-_Judgment.pdf text is 191,332 chars. This will increase token use.
Completed Ebeneezer_Paul_Tagoe_-vs-_Fernando_Private_Ltd_-_1403994.2023_-_Judgment.pdf


## 7. Summary

In [13]:
print("=== PIPELINE SUMMARY ===")
print(f"Run ID: {run_id}")
print(f"Run mode: {RUN_MODE}")
print(f"Judgments selected: {len(judgment_paths)}")
print(f"Judgments processed: {len(results)}")
print(f"WS chars: {len(ws_text):,} / {MAX_WS_CHARS:,}")
print(f"Cache root: {config.cache_root}")
print("Dictionary source: existing controlled dictionary file")
print(f"WS tagging output: {'created' if ws_tagging is not None else 'not created'}")
print(f"WS tagging summary themes: {len(ws_tagging_summary.get('theme_presence_by_id', {})) if ws_tagging_summary else 0}")

for result in results:
    print()
    print(f"Judgment: {result['judgment_path'].name}")
    print(f"Case slug: {result['case_slug']}")
    print(f"Case: {result['case_name']}")
    print(f"Judgment signals: {result['judgment_signals']}")
    print(f"Repair attempts: {result['repair_attempts']}")
    print(f"Compressed clusters: {result['compressed_clusters']}")
    if result['output_files']:
        print("Output files:")
        for label, output_path in result['output_files'].items():
            print(f"  {label}: {output_path}")

if not RUN_LLM:
    print("Model execution skipped. Set RUN_LLM = True in section 1 to run the selected flow.")

=== PIPELINE SUMMARY ===
Run ID: 20260502_071343
Run mode: debug
Judgments selected: 1
Judgments processed: 1
WS chars: 65,472 / 80,000
Cache root: /home/hello/Projects/Calibrator/output/cache
Dictionary source: existing controlled dictionary file
WS tagging output: created
WS tagging summary themes: 20

Judgment: Ebeneezer_Paul_Tagoe_-vs-_Fernando_Private_Ltd_-_1403994.2023_-_Judgment.pdf
Case slug: Ebeneezer_Paul_Tagoe_-vs-_Fernando_Private_Ltd_-_1403994.2023_-_Judgment
Case: Tagoe v Fernando Private Ltd
Judgment signals: 14
Repair attempts: 0
Compressed clusters: 0
Output files:
  Extracted Judgment: /home/hello/Projects/Calibrator/output/extracted_text/20260502_071343_Ebeneezer_Paul_Tagoe_-vs-_Fernando_Private_Ltd_-_1403994.2023_-_Judgment_text.txt
  Raw Calibration: /home/hello/Projects/Calibrator/output/calibration_raw/20260502_071343_Ebeneezer_Paul_Tagoe_-vs-_Fernando_Private_Ltd_-_1403994.2023_-_Judgment_calibration_raw.json
  Validated Calibration: /home/hello/Projects/Calibra